# Matryoshka Representations: Jointly Trained Nested Subspaces — companion notebook

The **narrative** half of the `matryoshka-nested-representations` topic. One source of truth:
`matryoshka_nested_representations.py` owns every number; this notebook imports it and walks the
topic section by section, running each assert so the claims render as executed output.

Run from the repo root:
```
uv run --with numpy --with scipy --with scikit-learn --with jupyter \
    jupyter execute notebooks/matryoshka-nested-representations/01_matryoshka_nested_representations.ipynb
```

In [ ]:
import sys, pathlib
_slug, _mod = "matryoshka-nested-representations", "matryoshka_nested_representations.py"
_cands = [pathlib.Path.cwd(), pathlib.Path.cwd() / "notebooks" / _slug]
_cands += [p / "notebooks" / _slug for p in pathlib.Path.cwd().parents]
for _c in _cands:
    if (_c / _mod).exists():
        sys.path.insert(0, str(_c)); break

import numpy as np
from matryoshka_nested_representations import (
    GRANULARITIES, structured_embeddings, pca_basis, pca_embedding, random_rotation,
    prefix_recon_error, eckart_young_rankm, joint_mrl_loss,
    prefix_recall, full_truth, funnel_retrieval, grid_table, finance_demo,
    test_linear_mrl_equals_pca, test_weight_invariance, test_prefix_recall_monotone,
    test_nested_beats_rotated, test_funnel_retrieval, test_sklearn_crosscheck, test_finance_funnel,
)
print("imported matryoshka_nested_representations")

## 1. The Matryoshka objective

A Matryoshka embedding $z\in\mathbb R^d$ is trained so that every prefix $z_{1:m}$, for
$m\in\mathcal M=\{d, d/2, d/4,\dots\}$, is itself usable. The objective is a weighted sum of
per-granularity losses,
$$ \mathcal L = \sum_{m\in\mathcal M} c_m \,\ell\big(z_{1:m}\big). $$
We study the linear, squared-reconstruction case, where $\ell$ measures how well the rank-$m$ prefix
reconstructs the data — and find that the jointly optimal basis is one we already know.

## 2. Linear Matryoshka is PCA

PCA's top-$m$ subspaces are **nested** — top-$m \subset$ top-$(m+1)$ — and each is the Eckart–Young
rank-$m$ optimum. So the single eigenvalue-ordered basis simultaneously minimizes *every* prefix
reconstruction term, hence the weighted sum for any positive $c_m$. The check below confirms the
PCA prefix error equals the rank-$m$ optimum to the decimal at every granularity.

In [ ]:
test_linear_mrl_equals_pca()

## 3. The optimum is weight-invariant

Because PCA attains *each* per-$m$ optimum, the joint loss equals the (unimprovable) weighted sum of
the per-$m$ optima for any positive weighting — so the optimal basis is PCA regardless of how the
granularities are weighted.

In [ ]:
test_weight_invariance()

## 4. Prefixes degrade gracefully — if nested

For the nested (PCA-ordered) embedding, prefix-$m$ retrieval recall is monotone in $m$ and reaches
the full-width recall. The early coordinates carry the structure, so a short prefix already retrieves
well.

In [ ]:
test_prefix_recall_monotone()

## 5. Nesting must be trained

Here is the catch that makes Matryoshka non-trivial. Take the same embedding and apply a random
rotation: every full-width distance is preserved exactly, so full-width recall is unchanged — but the
information is smeared across all coordinates, so a short prefix now retrieves no better than chance.
Nesting is a property of the *training*, not a free consequence of having an embedding.

In [ ]:
test_nested_beats_rotated()

## 6. Adaptive (funnel) retrieval

The payoff: shortlist candidates with a cheap short prefix, then rerank the shortlist at full width.
Recall close to exhaustive full-width search, at a fraction of the scoring cost.

In [ ]:
test_funnel_retrieval()

## 7. Cross-check against scikit-learn

Our nested basis is PCA, so its per-$m$ reconstruction matches `sklearn.decomposition.PCA`.

In [ ]:
test_sklearn_crosscheck()

## 8. The grid table and the finance case study

`grid_table()` is the block `MatryoshkaLaboratory.tsx` mirrors; `finance_demo()` is the $d=1536$
headline. Watch the nested recall hold up where the rotated one collapses, the nested reconstruction
sit exactly on the Eckart–Young optimum, and the funnel reach near-full recall cheaply.

In [ ]:
tbl = grid_table()
print(f"{'m':>6}{'nested_rec':>12}{'rotated_rec':>13}{'nested_recon':>14}{'optimum':>10}{'random_recon':>14}")
for r, c in zip(tbl['recall'], tbl['recon']):
    print(f"{r['m']:>6}{r['nested']:>12.4f}{r['rotated']:>13.4f}{c['nested']:>14.1f}{c['optimum']:>10.1f}{c['random']:>14.1f}")
print("funnel Pareto:", ", ".join(f"sl={f['shortlist']}->{f['recall']*100:.0f}%@{f['cost']*100:.1f}%" for f in tbl['funnel']))

In [ ]:
test_finance_funnel()

## What we verified

Every claim ran as an assert: linear Matryoshka equals PCA (one ordered basis hits every rank-$m$
optimum), weight invariance, prefix-recall monotonicity, the trained-nesting gap (nested vs rotated),
funnel retrieval, the scikit-learn cross-check, and the finance headline. The numbers above are the
ones the MDX and the viz cite.

In [ ]:
print('All Matryoshka claims verified against matryoshka_nested_representations.py')